# Minor Project 4: Predictive Classification using Random Forest Ensembles

**Dataset:** Breast Cancer Wisconsin (Diagnostic) dataset from Kaggle  
**Target:** Predict whether a tumor is benign or malignant  
**Positive class:** Malignant (`M = 1`)  
**Algorithm:** Random Forest Classifier

This notebook contains the complete project workflow, technical report, visualizations, evaluation metrics, conclusion, limitations, and possible improvements as required by the assignment.

# Importing Libraries


In [ ]:



import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully.")


In [ ]:




from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve
)

print("Machine learning libraries imported successfully.")


## Loading the Dataset

In [ ]:



import zipfile
import os
import pandas as pd

zip_path = '/content/archive.zip'

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/dataset')

print(os.listdir('/content/dataset'))



In [ ]:

# Loading the Kaggle Dataset

df = pd.read_csv('/content/dataset/data.csv')

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

display(df.head())


In [ ]:
# Data Cleaning
df = df.drop(columns=['Unnamed: 32', 'id'], errors='ignore')

print("Dataset cleaned successfully!")
print("Dataset Shape:", df.shape)
print("Missing Values:", df.isnull().sum().sum())
print("\nDiagnosis Counts:")
print(df['diagnosis'].value_counts())


In [ ]:

# Dataset Information and Missing Values

print("Column Names:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDataset Information:")
df.info()


**Observation:**

All required libraries were imported successfully. Pandas and NumPy are used for data handling and numerical calculations. Matplotlib and Seaborn are used for visualization, while Scikit-learn provides the tools required for classification and model evaluation.

**Observation:**

The dataset was loaded successfully. It contains breast tumor measurements and a target column for classification. The first five rows were displayed to understand the structure and features of the dataset.

## Understanding the Dataset

In [ ]:

# Display Dataset Information

print("Dataset Information:")
df.info()

print("\nDataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nStatistical Summary:")
df.describe()


**Observation:**

The dataset information shows the number of rows, columns, data types, and non-null values. The statistical summary provides information about the numerical features, including their mean, standard deviation, minimum, and maximum values. This helps us understand the dataset before preprocessing.

## Checking Missing Values

In [ ]:

# Check Missing Values

print("Missing values in each column:")
print(df.isnull().sum())

print("\nTotal missing values in the dataset:")
print(df.isnull().sum().sum())


**Observation:**

The missing values in each column were checked. This step helps identify incomplete data before model training. The results show the number of missing values present in the dataset.

## Checking Class Distribution

In [ ]:

# Check Target Class Distribution

print("Target Class Distribution:")
print(df["diagnosis"].value_counts())

print("\nTarget Class Percentages:")
print(df["diagnosis"].value_counts(normalize=True) * 100)


**Observation:** The dataset contains 569 records. Benign cases account for 62.74%, while malignant cases account for 37.26%. Therefore, the dataset has a moderate class imbalance, which should be considered when evaluating the model using precision, recall, F1-score, and ROC-AUC.

# Visualize Class Distribution

In [ ]:

# Distribution of Tumor Classes

class_counts = df["diagnosis"].value_counts()

plt.figure(figsize=(6, 4))
plt.bar(
    ["Benign", "Malignant"],
    [class_counts.get("B", 0), class_counts.get("M", 0)]
)

plt.title("Distribution of Tumor Classes")
plt.xlabel("Tumor Class")
plt.ylabel("Count")
plt.show()



**Observation:**

The target class distribution was examined to understand the number of benign and malignant tumor cases. The bar chart helps visualize the distribution of both classes and identify whether the dataset is imbalanced.

In [ ]:
# Feature Distribution Analysis
df[['radius_mean', 'texture_mean', 'area_mean']].hist(
    figsize=(10, 5), bins=20
)
plt.suptitle('Distribution of Important Features')
plt.tight_layout()
plt.show()


## Correlation Heatmap

In [ ]:

# Correlation Heatmap

plt.figure(figsize=(14, 10))

correlation = df.corr(numeric_only=True)

sns.heatmap(
    correlation,
    cmap="coolwarm",
    annot=False
)

plt.title("Correlation Heatmap of Dataset Features")
plt.show()


**Observation:**

The correlation heatmap shows the relationships between numerical features in the dataset. Values closer to +1 indicate a strong positive relationship, while values closer to -1 indicate a strong negative relationship. Values near 0 indicate a weak linear relationship. This helps us understand feature relationships before model training.

## Data Preprocessing: Separating Features and Target

In [ ]:

# Separate Features and Target

X = df.drop(columns=["diagnosis"])
y = df["diagnosis"].map({"B": 0, "M": 1})

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Features and target separated successfully!")



**Observation:**

The dataset was divided into features (X) and target (y). The features contain tumor measurements used for prediction, while the target represents the tumor class, where 0 indicates benign and 1 indicates malignant. This separation prepares the data for supervised machine learning.

## Splitting Training and Testing Data

In [ ]:

# Split Dataset into Training and Testing Sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)
print("Training target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)


**Observation:**

The dataset was divided into 80% training data and 20% testing data. The training data is used to build the Random Forest model, while the testing data is used to evaluate its performance on unseen examples. Stratified splitting helps preserve the class proportions in both sets.

### Preprocessing decisions

- **Missing values:** The dataset was checked for null values. The extra `Unnamed: 32` column and the `id` identifier column were removed. The extra column is not a useful predictor, and the ID is only an identifier.
- **Encoding:** All input features are numerical. The target column was encoded as `B = 0` (benign) and `M = 1` (malignant).
- **Feature scaling:** StandardScaler was **not used**. Random Forest is based on decision-tree splits and does not depend on distances or gradient-based optimization, so scaling numerical features is generally unnecessary.
- **Train-test split:** An 80:20 split was used. `stratify=y` keeps the benign/malignant class proportions approximately consistent in the training and test sets.

## Model Development: Random Forest Classifier

In [ ]:

# Initialize and Train Random Forest Classifier

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

model.fit(X_train, y_train)

print("Random Forest model trained successfully.")


**Observation:**

The Random Forest Classifier was trained using the training dataset. The model uses 100 decision trees and a maximum depth of 5 to help control model complexity. Random Forest combines predictions from multiple decision trees to perform classification.

### Why these hyperparameters?

- **`n_estimators=100`:** Uses 100 decision trees. This is a practical starting point that usually gives a stable ensemble without excessive computation.
- **`max_depth=5`:** Limits each tree's depth to control complexity and reduce the risk of overfitting. A different depth or number of trees could perform differently, so hyperparameter tuning is a useful future improvement.
- **`random_state=42`:** Makes the random choices reproducible so the experiment can be repeated consistently.

## Making Predictions

In [ ]:

# Predict Tumor Classes

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Actual values:")
print(y_test.head(10).tolist())

print("\nPredicted values:")
print(y_pred[:10].tolist())


**Observation:**

The trained Random Forest model predicted the tumor classes for the testing dataset. The actual and predicted values can be compared to understand how well the model classifies unseen data.

## Model Evaluation

In [ ]:

# Calculate Model Evaluation Metrics

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-Score:", round(f1, 4))
print("ROC-AUC Score:", round(auc, 4))


**Observation:**

The Random Forest model was evaluated using accuracy, precision, recall, F1-score, and ROC-AUC. Accuracy measures overall correctness, precision measures the reliability of positive predictions, recall measures how many actual positive cases were identified, and the F1-score balances precision and recall. ROC-AUC measures the model's ability to distinguish between the two classes.

### Interpreting precision, recall, and F1-score

Here, **malignant is the positive class**.

- **Precision:** Of all cases predicted as malignant, the proportion that were actually malignant. High precision means fewer benign cases are incorrectly flagged as malignant.
- **Recall (sensitivity):** Of all actual malignant cases, the proportion correctly detected. In a medical screening context, low recall would mean more malignant cases are missed, so recall is especially important.
- **Trade-off:** Increasing the threshold for predicting malignant may reduce false alarms but can also miss more malignant cases. Lowering it may catch more malignant cases but can increase false positives.
- **F1-score:** The harmonic mean of precision and recall. It gives a single measure that balances both, which is useful when class counts differ.
- **ROC-AUC:** Summarizes how well the model ranks malignant cases above benign cases across different classification thresholds.

The dataset has more benign than malignant cases (about 62.7% versus 37.3%), so accuracy alone is not enough; precision, recall, F1-score, and ROC-AUC provide a more complete evaluation.

## Confusion Matrix

In [ ]:

# Plot Confusion Matrix

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Benign", "Malignant"],
    yticklabels=["Benign", "Malignant"]
)

plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title("Random Forest Confusion Matrix")
plt.show()

print("Confusion Matrix:")
print(cm)


**Observation:**

The confusion matrix compares actual tumor classes with predicted classes. It shows correct predictions and misclassifications for both classes. This helps identify where the model performs well and where prediction errors occur.

**How to read the confusion matrix:** Rows are the actual classes and columns are the predicted classes. With malignant encoded as the positive class:
- **True negative (TN):** Benign correctly predicted as benign.
- **False positive (FP):** Benign incorrectly predicted as malignant.
- **False negative (FN):** Malignant incorrectly predicted as benign.
- **True positive (TP):** Malignant correctly predicted as malignant.

False negatives are particularly important to monitor in this dataset because they represent malignant cases that the model did not identify.

## ROC-AUC Curve

In [ ]:

# Plot ROC Curve

fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"ROC-AUC = {auc_score:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Random Forest")
plt.legend()
plt.show()


**Observation:**

The ROC curve shows the relationship between the true positive rate and false positive rate at different classification thresholds. A higher ROC-AUC score generally indicates better ability to distinguish between the two classes.

## Feature Importance

In [ ]:

# Identify Important Features

importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

plt.figure(figsize=(9, 6))
importance.head(10).sort_values().plot(kind="barh")

plt.xlabel("Feature Importance")
plt.ylabel("Features")
plt.title("Top 10 Important Features")
plt.show()

print("Top 10 Important Features:")
print(importance.head(10))


**Observation:**

The feature importance chart identifies the features that contributed most to the Random Forest model's predictions. These importance scores help us understand which tumor measurements were most influential in the classification process.

## Conclusion

The project, **Predictive Classification using Random Forest Ensembles**, uses the Breast Cancer Wisconsin dataset downloaded from Kaggle. The dataset contains 569 records and 30 numerical features used for prediction.

The data was cleaned, explored, and divided into 80% training data and 20% testing data. A Random Forest classifier was trained using 100 decision trees and a maximum depth of 5.

The model achieved **97.37% accuracy**, **100% precision**, **92.86% recall**, an **F1-score of 96.30%**, and a **ROC-AUC score of 0.995** on the test set.

The confusion matrix, ROC curve, correlation heatmap, and feature-importance chart helped evaluate and understand the model. Random Forest performed well on this test set, but further testing on independent datasets is needed before considering real-world medical use.

# Technical Report

## 1. Objective
The objective is to apply a Random Forest ensemble to a real classification problem and evaluate it using industry-standard classification metrics. This project predicts whether a breast tumor is benign or malignant.

## 2. Dataset
The Breast Cancer Wisconsin (Diagnostic) dataset was obtained from Kaggle. It contains **569 records**, a diagnosis target (`B` for benign and `M` for malignant), and **30 numerical predictor features** after removing the identifier and extra empty column. The target classes are approximately **62.74% benign (357 cases)** and **37.26% malignant (212 cases)**.

## 3. Exploratory Data Analysis
The notebook examines dataset dimensions, column types, summary statistics, missing values, class distribution, selected feature distributions, and a correlation heatmap. The class distribution shows that benign cases are more common than malignant cases. The heatmap helps reveal correlations among numerical measurements; strongly correlated features may carry overlapping information.

## 4. Data Cleaning and Preprocessing
The `id` column was removed because it is an identifier rather than a tumor measurement. The `Unnamed: 32` column was also removed because it is an extra empty column. Missing values were checked. The input features are already numerical, so no feature encoding was required. The diagnosis target was mapped to `B = 0` and `M = 1`. The data was split into 80% training data and 20% test data using `random_state=42` and stratification. Feature scaling was skipped because Random Forest uses decision-tree splits and is not distance-based.

## 5. Model Development
A `RandomForestClassifier` was trained with `n_estimators=100`, `max_depth=5`, and `random_state=42`. One hundred trees provide an ensemble of predictions, while a maximum depth of five limits tree complexity and helps reduce overfitting. These are reasonable baseline settings; a systematic search could identify better settings.

## 6. Results and Evaluation
The test-set results from this run were:

| Metric | Score |
|---|---:|
| Accuracy | 0.9737 (97.37%) |
| Precision | 1.0000 (100.00%) |
| Recall | 0.9286 (92.86%) |
| F1-score | 0.9630 (96.30%) |
| ROC-AUC | 0.9950 (99.50%) |

Accuracy indicates the overall proportion of correct predictions. Precision of 100% means that, in this test split, every case predicted as malignant was actually malignant. Recall of 92.86% means that the model detected about 92.86% of the actual malignant cases; some malignant cases were missed. The F1-score balances precision and recall. The ROC-AUC score indicates strong separation between benign and malignant cases across thresholds. These scores describe this particular test split and do not guarantee the same performance on new data.

The confusion matrix visualizes true negatives, false positives, false negatives, and true positives. The ROC curve plots true positive rate against false positive rate. The feature-importance chart ranks the input measurements by the importance values learned by the Random Forest model. Feature importance describes model use, not proof of cause.

## 7. Conclusion
The Random Forest model performed strongly on the held-out test set, achieving 97.37% accuracy and a 0.9950 ROC-AUC. Precision was 100%, while recall was 92.86%, showing that the model still missed some malignant cases. Recall and false negatives deserve particular attention for this task.

## 8. Limitations and Future Improvements
- The evaluation uses one train-test split and a relatively small dataset; performance may vary on other samples.
- The dataset may not represent all hospitals, populations, or measurement conditions.
- Feature importance does not establish a causal relationship.
- Future work could use cross-validation, tune `n_estimators` and `max_depth`, compare Gradient Boosting or other classifiers, and evaluate the model on an independent dataset.
- This educational model should not be used for real medical diagnosis without independent clinical validation and appropriate expert oversight.

In [ ]:
# Final summary for the technical report
print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["Benign", "Malignant"],
    digits=4,
    zero_division=0
))